# Phase B Teacher — RoBERTa-large on FiNER-ORD (Colab T4)

Runs `configs/phase_b_teacher.yaml` (3 seeds × up to 30 epochs, fp16, early stopping).

T4 wall-clock estimate: ~45–90 min per seed with early stopping. Budget a Colab Pro session; free tier may time out mid-run.

**Before you start:**
- Runtime → Change runtime type → T4 GPU.
- Repo must be reachable from Colab (Drive mount or GitHub clone in the next cell).
- `WANDB_API_KEY` must be set (Colab secret preferred, inline fallback).

In [1]:
!nvidia-smi

zsh:1: command not found: nvidia-smi


In [2]:
from google.colab import runtime
runtime.unassign()

ModuleNotFoundError: No module named 'google.colab'

In [ ]:
import os

# Remove existing FER_NLP directory if it exists to ensure a clean clone
if os.path.exists('FER_NLP'):
    !rm -rf FER_NLP
    print('Removed existing FER_NLP directory.')

# GitHub clone
print('Attempting to clone the repository...')
!git clone https://github.com/dayan-battulga/FER_NLP.git

# Verify clone and list contents
if os.path.exists('FER_NLP'):
    print('\nSuccessfully cloned FER_NLP. Listing contents:')
    !cd FER_NLP && ls -F
else:
    print('\nFailed to clone FER_NLP.')

# Go into project directory
%cd FER_NLP
!ls

# Install packages
!pip install -q -r requirements.txt

In [ ]:
# On Colab, do NOT reinstall torch. Colab's prebuilt torch already has CUDA
# support for the T4, and pinning torch==2.3.1 from PyPI can silently swap in
# a CPU-only wheel that makes training run on CPU (hours -> days).
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    !pip install -q \
        "transformers==4.44.2" \
        "tokenizers==0.19.1" \
        "datasets==2.21.0" \
        "accelerate==0.33.0" \
        "seqeval==1.2.2" \
        "pytorch-crf==0.7.2" \
        "wandb==0.17.7" \
        "pyyaml==6.0.2"
else:
    !pip install -q -r requirements.txt

# Gate: CUDA torch must still be active before any long-running training.
import torch
print("torch:", torch.__version__,
      "| cuda available:", torch.cuda.is_available(),
      "| device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")
assert torch.cuda.is_available(), "No CUDA. Restart runtime and re-run this cell without reinstalling torch."

In [ ]:
!wandb login

In [ ]:
# Smoke test first (~3-5 min) to catch plumbing issues before the long run.
!python -m src.train --config configs/smoke_test.yaml

In [ ]:
# Phase B teacher - 3 seeds x up to 30 epochs. Long-running.
!python -m src.train --config configs/phase_b_teacher.yaml

In [ ]:
import pandas as pd
df = pd.read_csv('results/results.csv')
phase_b = df[df['notes'].str.contains('phase_b_teacher', na=False)]
print(phase_b.to_string(index=False))
print()
print('Mean entity F1:', phase_b['test_entity_f1'].astype(float).mean())
print('Std  entity F1:', phase_b['test_entity_f1'].astype(float).std())

In [ ]:
import json
with open('results/phase_b_teacher_aggregate.json') as f:
    agg = json.load(f)
print(json.dumps(agg['test_metrics'], indent=2))